# Yambda 09 · SASRec v2: align training with the task (Colab GPU)

SASRec v1 (notebook 08): val recall@10 = **0.0268**, below iALS (0.0346) and BPR (0.0334), but with
much higher coverage (9.0% vs 5.5%) and less popular recommendations. Three hypotheses for the gap,
each from something we learned earlier, tested one at a time (ablation):

| Variant | Change | Why |
|---|---|---|
| A | loss only on **first listens** | v1 learned to predict *replays* (71% of listens), which the evaluator masks out. Our targets are discoveries. |
| B | A + training windows end in the **last 30 days** | iALS was best with a 3-day half-life: recent behaviour matters most. |
| C | B + **8192 negatives** (was 2048) | a less noisy approximation of the full softmax over 160k tracks |

**Before running:** upload the small `colab_src.zip` (updated code, ~50 KB) to `MyDrive/music_recsys/`
next to `colab_bundle.zip`. Runtime → T4 GPU. ~10 min per variant, ~35 min total.

In [2]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/music_recsys"
!rm -rf /content/music_recsys && mkdir -p /content/music_recsys
!unzip -q "{DRIVE}/colab_bundle.zip" -d /content/music_recsys
!unzip -q -o "{DRIVE}/colab_src.zip" -d /content/music_recsys      # overwrite with the updated code

import sys
sys.path.insert(0, "/content/music_recsys/src")
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime!)")

Mounted at /content/drive
torch 2.11.0+cu130 | GPU: Tesla T4


In [4]:
import json, time
import pandas as pd
import matplotlib.pyplot as plt
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate, paired_bootstrap, artist_novelty
from musicrec.yambda import load_item_artist
from musicrec.models.sasrec import SASRec

pd.set_option("display.float_format", "{:,.4f}".format)
split = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
OUT = config.PROJECT_ROOT / "reports" / "yambda"
config.YAMBDA_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ref = {k: pd.read_parquet(OUT / f"per_user_val_{k}.parquet") for k in ["pop7d", "ials", "bpr"]}
item_artist = load_item_artist()
new_artist = artist_novelty(split, item_artist)

BASE = dict(dim=128, max_len=200, n_layers=2, n_heads=2, dropout=0.2, lr=1e-3,
            batch_size=128, p_last=0.5, steps_per_epoch=400, epochs=30, patience=3)
VARIANTS = {
    "A first-listen loss":        dict(n_neg=2048, loss_on="first"),
    "B + recent 30d windows":     dict(n_neg=2048, loss_on="first", recent_days=30),
    "C + 8192 negatives":         dict(n_neg=8192, loss_on="first", recent_days=30),
}

In [7]:
runs, models = {}, {}
for label, kw in VARIANTS.items():
    m = SASRec(**BASE, **kw)
    print(f"\n== {label}: {m.name}")
    t = time.time(); m.fit(split)
    r = evaluate(m, split, return_per_user=True)
    r["segments"] = {lab: evaluate(m, split, targets=split.targets[mask])["recall@10"]
                     for lab, mask in [("known-artist", ~new_artist), ("new-artist", new_artist)]}
    r["minutes"], r["best_epoch"] = (time.time() - t) / 60, m.best_epoch_
    runs[label], models[label] = r, m
    print(f"recall@10={r['recall@10']:.4f}  best epoch {m.best_epoch_}  ({r['minutes']:.0f} min)")


== A first-listen loss: SASRec(d=128, L=200, layers=2, neg=2048, p_last=0.5, loss=first-listens)
epoch  1  loss=6.5876  (32s)  val recall@10=0.0042
epoch  2  loss=6.0784  (31s)  val recall@10=0.0037
epoch  3  loss=5.6143  (33s)  val recall@10=0.0069
epoch  4  loss=5.0668  (32s)  val recall@10=0.0103
epoch  5  loss=4.5319  (32s)  val recall@10=0.0137
epoch  6  loss=4.0663  (32s)  val recall@10=0.0172
epoch  7  loss=3.6701  (32s)  val recall@10=0.0189
epoch  8  loss=3.3647  (32s)  val recall@10=0.0221
epoch  9  loss=3.1339  (32s)  val recall@10=0.0244
epoch 10  loss=2.9564  (32s)  val recall@10=0.0230
epoch 11  loss=2.8063  (32s)  val recall@10=0.0255
epoch 12  loss=2.7010  (32s)  val recall@10=0.0258
epoch 13  loss=2.6139  (32s)  val recall@10=0.0236
epoch 14  loss=2.5374  (32s)  val recall@10=0.0244
epoch 15  loss=2.4734  (32s)  val recall@10=0.0228
early stop: no improvement in 3 epochs
recall@10=0.0258  best epoch 12  (12 min)

== B + recent 30d windows: SASRec(d=128, L=200, layers=

## Ablation table (v1 from notebook 08 for reference)

In [5]:
import shutil
shutil.copy(f"{DRIVE}/outputs/best_sasrec.json", OUT / "best_sasrec.json")
print("copied:", (OUT / "best_sasrec.json").exists())

copied: True


In [8]:
v1 = json.loads((OUT / "best_sasrec.json").read_text())
rows = {"v1 (notebook 08)": {**{k: v1["val"][k] for k in ["recall@10", "ndcg@10", "hit_rate@10", "coverage@10", "avg_pop@10"]},
                             "known-artist": v1["segments"]["known-artist targets"], "new-artist": v1["segments"]["new-artist targets"]}}
for label, r in runs.items():
    rows[label] = {**{k: r[k] for k in ["recall@10", "ndcg@10", "hit_rate@10", "coverage@10", "avg_pop@10"]},
                   "known-artist": r["segments"]["known-artist"], "new-artist": r["segments"]["new-artist"],
                   "best_epoch": r["best_epoch"], "minutes": r["minutes"]}
for key, name in [("ials", "iALS (tuned)"), ("bpr", "BPR (tuned)"), ("pop7d", "Popular 7d")]:
    rows[name] = {"recall@10": ref[key]["recall@10"].mean(), "ndcg@10": ref[key]["ndcg@10"].mean(),
                  "hit_rate@10": ref[key]["hit_rate@10"].mean()}
ablation = pd.DataFrame(rows).T
ablation.to_csv(OUT / "sasrec_ablation.csv")
ablation

,recall@10,ndcg@10,hit_rate@10,coverage@10,avg_pop@10,known-artist,new-artist,best_epoch,minutes
v1 (notebook 08),0.0268,0.0257,0.1305,0.0903,455.4357,0.0253,0.0152,NaN,NaN
A first-listen loss,0.0258,0.0236,0.1205,0.1148,348.7346,0.0247,0.0146,12.0000,11.7786
B + recent 30d windows,0.0213,0.0198,0.1032,0.1288,350.0406,0.0212,0.0121,12.0000,11.6922
C + 8192 negatives,0.0220,0.0206,0.1067,0.0980,390.5959,0.0211,0.0116,3.0000,6.2148
iALS (tuned),0.0346,0.0322,0.1671,NaN,NaN,NaN,NaN,NaN,NaN
BPR (tuned),0.0334,0.0306,0.1627,NaN,NaN,NaN,NaN,NaN,NaN
Popular 7d,0.0235,0.0224,0.1062,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
best = max(runs, key=lambda k: runs[k]["recall@10"])
print("best variant:", best)
pd.DataFrame({f"{best} vs {k}": paired_bootstrap(v, runs[best]["per_user"]) for k, v in ref.items()}).T[
    ["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]]

best variant: A first-listen loss


,a,b,diff,rel_diff,ci_low,ci_high,p(b<=a)
A first-listen loss vs pop7d,0.0235,0.0258,0.0023,0.0993,-0.0011,0.0057,0.0860
A first-listen loss vs ials,0.0346,0.0258,-0.0088,-0.2535,-0.0120,-0.0057,1.0000
A first-listen loss vs bpr,0.0334,0.0258,-0.0076,-0.2263,-0.0109,-0.0042,1.0000


In [10]:
# Complementarity: how often does SASRec find something iALS misses (and vice versa)?
m = ref["ials"].merge(runs[best]["per_user"], on="user_idx", suffixes=("_ials", "_sas"))
hi, hs = m["hit_rate@10_ials"] > 0, m["hit_rate@10_sas"] > 0
pd.Series({"hit iALS": hi.mean(), "hit SASRec": hs.mean(), "hit either (oracle)": (hi | hs).mean(),
           "SASRec only": (hs & ~hi).mean(), "iALS only": (hi & ~hs).mean(),
           "corr of per-user recall": m["recall@10_ials"].corr(m["recall@10_sas"])})

,0
hit iALS,0.1671
hit SASRec,0.1205
hit either (oracle),0.2270
SASRec only,0.0599
iALS only,0.1066
corr of per-user recall,0.3184


In [11]:
# Save the best variant and copy outputs back to Drive
bm = models[best]
torch.save({"state_dict": bm.module_.state_dict(), "params": bm.params(), "best_epoch": bm.best_epoch_},
           OUT / "sasrec_v2_val.pt")
runs[best]["per_user"].to_parquet(OUT / "per_user_val_sasrec_v2.parquet", index=False)
(OUT / "best_sasrec_v2.json").write_text(json.dumps({
    "variant": best, "params": bm.params(), "best_epoch": bm.best_epoch_,
    "val": {k: v for k, v in runs[best].items() if k not in ("per_user",)}}, indent=2, default=float))
!mkdir -p "{DRIVE}/outputs_v2" && cp {OUT}/sasrec_v2_val.pt {OUT}/per_user_val_sasrec_v2.parquet {OUT}/best_sasrec_v2.json {OUT}/sasrec_ablation.csv "{DRIVE}/outputs_v2/"
!ls -la "{DRIVE}/outputs_v2"

total 82082
drwx------ 2 root root     4096 Oct  8 15:45 .
drwx------ 4 root root     4096 Oct  8 15:44 ..
-rw------- 1 root root      915 Oct  8 15:45 best_sasrec_v2.json
-rw------- 1 root root    64564 Oct  8 15:45 per_user_val_sasrec_v2.parquet
-rw------- 1 root root     1051 Oct  8 15:45 sasrec_ablation.csv
-rw------- 1 root root 83975518 Oct  8 15:45 sasrec_v2_val.pt
